In [5]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph,START,END
from typing import TypedDict

In [4]:
load_dotenv()
llm=ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite-preview")

In [6]:
# Defining State
class LLMState(TypedDict):
    
    question:str
    answer:str

In [14]:
def llm_node(state:LLMState)->LLMState:
    
    # Extract Question
    question=state['question']
    
    # Form PromptTemplate
    prompt=PromptTemplate(
        template="""You are a very funny guy answer the following question in atmost 150 words in fun tone \n question:{question} """,
        input_variables=['question']
        )
    
    # Define Parser
    parser=StrOutputParser()
    
    # Forming Chain
    chain=prompt | llm | parser 
    
    # Call LLM
    answer=chain.invoke({'question':question})
    
    # Update Answer
    state['answer']=answer
    
    # Return State
    return state  
    

In [15]:
# Define Graph
graph=StateGraph(LLMState)

# Add Nodes
graph.add_node('llm_node',llm_node)

# Add Edges
graph.add_edge(START,'llm_node')
graph.add_edge('llm_node',END)

# Compile Graph
workflow=graph.compile()

In [16]:
final_output=workflow.invoke({'question':"What is Collge Life?"})
print(final_output)

{'question': 'What is Collge Life?', 'answer': 'College life is basically a high-stakes social experiment where you pay thousands of dollars to learn that “tomorrow” is a myth and caffeine is a food group. \n\nIt’s the only time in your life where eating cold leftover pizza at 3:00 AM while crying over a PowerPoint slide is considered a "productive Tuesday." You’ll spend four years perfecting the art of walking into a class you haven\'t attended in months, looking the professor dead in the eye, and nodding like you totally read the syllabus. \n\nYou’ll acquire a degree, a questionable sleep schedule, and a personality composed entirely of inside jokes and existential dread. It’s a beautiful, chaotic blur of bad decisions, expensive textbooks that become expensive doorstops, and friendships forged in the fires of group projects where you do all the work. Enjoy the chaos!'}
